# Generate Attribution Graphs

This notebook uses `circuit-tracer` to run **Gemma-2-2B** and **Llama-3.2-1B**
on each prompt and saves pruned attribution graphs to `data/graphs/<model>/`.

**Prerequisites:**
```bash
pip install -r requirements.txt
huggingface-cli login   # both models require accepting their licences on HF
```

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

import torch
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
with open(ROOT / "data" / "prompts.json") as f:
    PROMPTS = json.load(f)

print(f"Loaded {len(PROMPTS)} prompts")
for p in PROMPTS:
    print(f"  [{p['id']:20s}] {p['prompt']}")

## Model configuration

| Model          | HF repo                        | Circuit-tracer scan type |
|----------------|--------------------------------|--------------------------|
| Gemma-2-2B     | `google/gemma-2-2b`            | `gemma`                  |
| Llama-3.2-1B   | `meta-llama/Llama-3.2-1B`      | `llama`                  |

In [ ]:
MODEL_CONFIGS = [
    {
        "name": "gemma",
        "hf_repo": "google/gemma-2-2b",
        "scan_type": "gemma",
        "output_dir": ROOT / "data" / "graphs" / "gemma",
    },
    {
        "name": "llama",
        "hf_repo": "meta-llama/Llama-3.2-1B",
        "scan_type": "llama",
        "output_dir": ROOT / "data" / "graphs" / "llama",
    },
]

PRUNE_THRESHOLD = 0.01
MAX_NODES = 500

for cfg in MODEL_CONFIGS:
    cfg["output_dir"].mkdir(parents=True, exist_ok=True)
    print(f"Output dir ready: {cfg['output_dir']}")

In [ ]:
# -----------------------------------------------------------------------
# Import circuit-tracer
# API reference: https://github.com/decoderesearch/circuit-tracer
# -----------------------------------------------------------------------
try:
    from circuit_tracer import CircuitTracer
    print("circuit-tracer imported successfully.")
except ImportError as e:
    print(f"[ERROR] circuit-tracer not installed: {e}")
    print("Install with: pip install circuit-tracer")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model_and_tracer(cfg: dict, device: str = DEVICE):
    print(f"Loading {cfg['hf_repo']} …")
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_repo"])
    model = AutoModelForCausalLM.from_pretrained(
        cfg["hf_repo"],
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto",
    )
    model.eval()
    return CircuitTracer(model, tokenizer, scan_type=cfg["scan_type"])


def run_tracing_for_model(cfg: dict, prompts: list[dict]) -> None:
    tracer = load_model_and_tracer(cfg)
    out_dir: Path = cfg["output_dir"]

    for entry in tqdm(prompts, desc=cfg["name"]):
        out_path = out_dir / f"{entry['id']}.json"
        if out_path.exists():
            print(f"  Skip (exists): {out_path.name}")
            continue
        try:
            graph = tracer.trace(
                entry["prompt"],
                prune_threshold=PRUNE_THRESHOLD,
                max_nodes=MAX_NODES,
            )
            graph_dict = graph.to_dict() if hasattr(graph, "to_dict") else graph
            graph_dict.setdefault("metadata", {})
            graph_dict["metadata"].update({
                "prompt": entry["prompt"],
                "prompt_id": entry["id"],
                "category": entry["category"],
                "model": cfg["name"],
            })
            with out_path.open("w") as f:
                json.dump(graph_dict, f, indent=2)
            print(f"  Saved {out_path.name}: "
                  f"{len(graph_dict.get('nodes', []))} nodes, "
                  f"{len(graph_dict.get('edges', []))} edges")
        except Exception as exc:
            print(f"  [ERROR] {entry['id']}: {exc}")

In [ ]:
# Loads each model in turn to avoid OOM on CPU.
for cfg in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f" Model: {cfg['name']}")
    print(f"{'='*60}")
    run_tracing_for_model(cfg, PROMPTS)

## Quick sanity check

In [ ]:
from src.graph_processor import GraphProcessor
from src.circuit_comparator import CircuitComparator

processor = GraphProcessor()

pid = PROMPTS[0]["id"]
try:
    ga = processor.load_graph(ROOT / "data" / "graphs" / "gemma" / f"{pid}.json")
    gb = processor.load_graph(ROOT / "data" / "graphs" / "llama" / f"{pid}.json")
    comparator = CircuitComparator(ga, gb)
    result = comparator.compare()

    print(comparator.feature_overlap_report())
    print(f"\nComposite similarity : {result['metrics']['composite_similarity']:.3f}")
    print(f"Gemma  — {ga.summary()}")
    print(f"Llama  — {gb.summary()}")
except FileNotFoundError as e:
    print(f"Graph file not found — run the tracing cells above first.\n{e}")

## Save comparison results for all prompts

In [ ]:
import pandas as pd

results_dir = ROOT / "results" / "analysis"
results_dir.mkdir(parents=True, exist_ok=True)

rows = []
for p in PROMPTS:
    ga_path = ROOT / "data" / "graphs" / "gemma" / f"{p['id']}.json"
    gb_path = ROOT / "data" / "graphs" / "llama" / f"{p['id']}.json"
    if not ga_path.exists() or not gb_path.exists():
        print(f"Skipping {p['id']} — missing files")
        continue

    ga = processor.load_graph(ga_path)
    gb = processor.load_graph(gb_path)
    comp = CircuitComparator(ga, gb).compare()

    m = comp["metrics"]
    rows.append({
        "prompt_id": p["id"],
        "category": p["category"],
        "composite_similarity": m["composite_similarity"],
        "jaccard": m["feature_overlap"]["jaccard"],
        "structural_similarity": m["structural_similarity"]["composite_similarity"],
        "layer_emd": m["layer_distribution"]["emd"],
        "shared_features": m["feature_overlap"]["shared"],
        "gemma_nodes": comp["summary_a"]["num_nodes"],
        "llama_nodes": comp["summary_b"]["num_nodes"],
    })

if rows:
    df = pd.DataFrame(rows)
    out_csv = results_dir / "comparison_summary.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved {len(df)} rows to {out_csv}")
    display(df.sort_values("composite_similarity", ascending=False))
else:
    print("No results to save — generate graphs first.")